In [1]:
# ============================================================
# Task 4-4 — Multi-Label Image Classification Training
# Import Required Libraries
# ============================================================

import os
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from sklearn.model_selection import train_test_split

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.20.0


In [2]:
# ============================================================
# Define Paths and Load labels.csv
# ============================================================

BASE_DIR = r"E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-4_MultiLabel_Service"
DATASET_DIR = os.path.join(BASE_DIR, "dataset")
IMAGES_DIR = os.path.join(DATASET_DIR, "multilabel_images")
LABELS_CSV = os.path.join(DATASET_DIR, "labels.csv")

print("Images folder:", IMAGES_DIR)
print("Labels file:", LABELS_CSV)

df = pd.read_csv(LABELS_CSV)

print("\nlabels.csv shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


Images folder: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-4_MultiLabel_Service\dataset\multilabel_images
Labels file: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-4_MultiLabel_Service\dataset\labels.csv

labels.csv shape: (40, 4)
Columns: ['filename', 'bike', 'bus', 'car']


,filename,bike,bus,car
0,ml_2_000.jpg,1,0,1
1,ml_2_001.jpg,1,0,1
2,ml_2_002.jpg,1,0,1
3,ml_2_003.jpg,1,0,1
4,ml_2_004.jpg,1,0,1


In [3]:
# ============================================================
# Verify Image Files Exist + Quick Label Distribution
# ============================================================

missing = []
for fn in df["filename"].tolist():
    p = os.path.join(IMAGES_DIR, fn)
    if not os.path.exists(p):
        missing.append(fn)

print("Total label rows:", len(df))
print("Missing image files:", len(missing))
if missing:
    print("First missing examples:", missing[:5])

print("\nLabel counts (how many images contain each label):")
print(df[["bike", "bus", "car"]].sum())

print("\nHow many labels per image (multi-label count):")
print(df[["bike", "bus", "car"]].sum(axis=1).value_counts().sort_index())


Total label rows: 40
Missing image files: 0

Label counts (how many images contain each label):
bike    34
bus     27
car     29
dtype: int64

How many labels per image (multi-label count):
2    30
3    10
Name: count, dtype: int64


In [4]:
# ============================================================
# Build tf.data Pipeline (load → resize → normalize → batch)
# ============================================================

IMG_SIZE = 224
BATCH_SIZE = 8
LABEL_COLS = ["bike", "bus", "car"]

# file paths + label vectors
file_paths = df["filename"].apply(lambda x: os.path.join(IMAGES_DIR, x)).values
labels = df[LABEL_COLS].values.astype("float32")

# train/val split
X_train, X_val, y_train, y_val = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42, shuffle=True
)

print("Train samples:", len(X_train))
print("Val samples:", len(X_val))
print("Label order:", LABEL_COLS)

def load_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_ds   = tf.data.Dataset.from_tensor_slices((X_val, y_val))

train_ds = train_ds.shuffle(buffer_size=len(X_train)).map(load_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
val_ds   = val_ds.map(load_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# sanity check one batch
for images, y in train_ds.take(1):
    print("Batch images shape:", images.shape)
    print("Batch labels shape:", y.shape)
    print("Example labels (first 5 rows):\n", y[:5].numpy())


Train samples: 32
Val samples: 8
Label order: ['bike', 'bus', 'car']
Batch images shape: (8, 224, 224, 3)
Batch labels shape: (8, 3)
Example labels (first 5 rows):
 [[1. 0. 1.]
 [1. 0. 1.]
 [1. 1. 0.]
 [1. 1. 1.]
 [1. 1. 1.]]


In [5]:
# ============================================================
# Build MobileNetV2 Multi-Label Model (Sigmoid Output)
# ============================================================

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # first train: frozen backbone

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(LABEL_COLS), activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="bin_acc", threshold=0.5),
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
# ============================================================
# Train Model (Frozen MobileNetV2)
# ============================================================

EPOCHS = 8

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 902ms/step - auc: 0.5753 - bin_acc: 0.5729 - loss: 0.7498 - val_auc: 0.5084 - val_bin_acc: 0.7500 - val_loss: 0.6866
Epoch 2/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 231ms/step - auc: 0.7040 - bin_acc: 0.7604 - loss: 0.5141 - val_auc: 0.4958 - val_bin_acc: 0.7083 - val_loss: 0.7946
Epoch 3/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 237ms/step - auc: 0.7219 - bin_acc: 0.7500 - loss: 0.5478 - val_auc: 0.4748 - val_bin_acc: 0.7083 - val_loss: 0.8206
Epoch 4/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 240ms/step - auc: 0.7781 - bin_acc: 0.7604 - loss: 0.5104 - val_auc: 0.4916 - val_bin_acc: 0.7083 - val_loss: 0.7546
Epoch 5/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 242ms/step - auc: 0.7528 - bin_acc: 0.8021 - loss: 0.4728 - val_auc: 0.5378 - val_bin_acc: 0.7500 - val_loss: 0.6952
Epoch 6/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 234ms/step - auc: 0.8088 - bin_acc: 0.8229 - loss: 0.4418 - val_auc: 0.5252 - val_bin_acc: 0.7083 - val_loss: 0.6537
Epoch 7/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 245ms/step - auc: 0.8172 - bin_a

In [7]:
# ============================================================
# Fine-Tune: Unfreeze Top Layers of MobileNetV2
# ============================================================

base_model.trainable = True

# Freeze lower layers, unfreeze top 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="bin_acc", threshold=0.5),
        tf.keras.metrics.AUC(name="auc")
    ]
)

print("Fine-tuning enabled.")

Fine-tuning enabled.


In [8]:
# ============================================================
# Train Model (Fine-Tuning Phase)
# ============================================================

FINE_TUNE_EPOCHS = 5

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS
)


Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - auc: 0.8243 - bin_acc: 0.7604 - loss: 0.4434 - val_auc: 0.6345 - val_bin_acc: 0.7083 - val_loss: 0.6257
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 303ms/step - auc: 0.8562 - bin_acc: 0.8125 - loss: 0.4145 - val_auc: 0.6387 - val_bin_acc: 0.7083 - val_loss: 0.6248
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 321ms/step - auc: 0.8398 - bin_acc: 0.8021 - loss: 0.4192 - val_auc: 0.6429 - val_bin_acc: 0.7083 - val_loss: 0.6244
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 310ms/step - auc: 0.8758 - bin_acc: 0.8542 - loss: 0.3821 - val_auc: 0.6471 - val_bin_acc: 0.7083 - val_loss: 0.6236
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 320ms/step - auc: 0.9208 - bin_acc: 0.8438 - loss: 0.3678 - val_auc: 0.6597 - val_bin_acc: 0.7500 - val_loss: 0.6221


In [9]:
# ============================================================
# Save Multi-Label Model
# ============================================================

MODEL_PATH = os.path.join(BASE_DIR, "model", "vehicle_multilabel.h5")

model.save(MODEL_PATH)

print("Model saved to:", MODEL_PATH)


Model saved to: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-4_MultiLabel_Service\model\vehicle_multilabel.h5
